In [2]:
import os
import sys
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)
# Set the parent directory as the current directory
os.chdir(parent_dir)

# filter out rare diseases from the phenotype list

In [3]:
import json
import re
from typing import List, Dict, Any, Set
from copy import deepcopy


def normalize_text(text: str) -> str:
    """
    Normalize text for comparison: lowercase, strip whitespace, and remove extra spaces
    """
    return re.sub(r'\s+', ' ', text.lower().strip())


def extract_disease_terms(patient_data: Dict[str, Any]) -> Set[str]:
    """
    Extract all disease terms from a patient's disease entities
    
    Args:
        patient_data: Patient data dictionary containing disease_entities
        
    Returns:
        Set of normalized disease terms
    """
    disease_terms = set()
    
    # Get disease entities from the patient data
    disease_entities = patient_data.get('disease_entities', [])
    
    for disease in disease_entities:
        if isinstance(disease, str):
            # Split disease name into individual terms
            disease_normalized = normalize_text(disease)
            # Add the full disease name
            disease_terms.add(disease_normalized)
            
            # Also add individual words from the disease name (for partial matching)
            words = disease_normalized.split()
            for word in words:
                if len(word) > 2:  # Only add words longer than 2 characters
                    disease_terms.add(word)
    
    return disease_terms


def contains_disease_term(phenotype_text: str, disease_terms: Set[str]) -> bool:
    """
    Check if a phenotype text contains any disease terms
    
    Args:
        phenotype_text: The phenotype text to check
        disease_terms: Set of disease terms to search for
        
    Returns:
        True if phenotype contains any disease term, False otherwise
    """
    phenotype_normalized = normalize_text(phenotype_text)
    
    # Check for exact disease name matches
    for disease_term in disease_terms:
        if disease_term in phenotype_normalized:
            return True
    
    return False


def remove_disease_phenotypes_from_patient(
    patient_data: Dict[str, Any], 
    verbose: bool = False
) -> Dict[str, Any]:
    """
    Remove phenotypes that contain rare disease names from a single patient's data
    
    Args:
        patient_data: Patient data dictionary
        verbose: Whether to print details about removed phenotypes
        
    Returns:
        Modified patient data with disease-containing phenotypes removed
    """
    # Create a deep copy to avoid modifying the original data
    modified_data = deepcopy(patient_data)
    
    # Extract disease terms from this patient's disease entities
    disease_terms = extract_disease_terms(patient_data)
    
    if not disease_terms:
        return modified_data
    
    # Process matched_phenotypes if they exist
    if 'matched_phenotypes' in modified_data:
        original_phenotypes = modified_data['matched_phenotypes']
        filtered_phenotypes = []
        removed_phenotypes = []
        
        for phenotype in original_phenotypes:
            phenotype_text = phenotype.get('phenotype', '')
            
            if phenotype_text and contains_disease_term(phenotype_text, disease_terms):
                removed_phenotypes.append(phenotype)
                if verbose:
                    print(f"  Removed phenotype: '{phenotype_text}'")
            else:
                filtered_phenotypes.append(phenotype)
        
        modified_data['matched_phenotypes'] = filtered_phenotypes
        
        if verbose and removed_phenotypes:
            print(f"  Total phenotypes removed: {len(removed_phenotypes)}")
            print(f"  Remaining phenotypes: {len(filtered_phenotypes)}")
    
    return modified_data


def remove_disease_phenotypes_from_dataset(
    data: Dict[str, Any], 
    verbose: bool = False
) -> Dict[str, Any]:
    """
    Remove phenotypes containing rare disease names from all patients in the dataset
    
    Args:
        data: Dictionary containing patient data with phenotypes and disease entities
        verbose: Whether to print detailed results for each patient
        
    Returns:
        Modified dataset with disease-containing phenotypes removed
    """
    modified_data = {}
    total_patients = len(data)
    patients_modified = 0
    total_phenotypes_removed = 0
    total_original_phenotypes = 0
    
    # First pass: count total original phenotypes
    for patient_data in data.values():
        total_original_phenotypes += len(patient_data.get('matched_phenotypes', []))
    
    if verbose:
        print(f"Processing {total_patients} patients...")
        print(f"Total original phenotypes across all patients: {total_original_phenotypes}")
        print("=" * 60)
    
    for patient_id, patient_data in data.items():
        if verbose:
            print(f"Patient ID: {patient_id}")
            
            # Count original phenotypes
            original_count = len(patient_data.get('matched_phenotypes', []))
            disease_entities = patient_data.get('disease_entities', [])
            
            print(f"  Original phenotypes: {original_count}")
            print(f"  Disease entities: {disease_entities}")
        
        # Process the patient
        modified_patient = remove_disease_phenotypes_from_patient(
            patient_data, verbose=verbose
        )
        
        # Count modifications
        modified_count = len(modified_patient.get('matched_phenotypes', []))
        phenotypes_removed = len(patient_data.get('matched_phenotypes', [])) - modified_count
        
        if phenotypes_removed > 0:
            patients_modified += 1
            total_phenotypes_removed += phenotypes_removed
        
        modified_data[patient_id] = modified_patient
        
        if verbose:
            print(f"  Final phenotypes: {modified_count}")
            print("-" * 60)
    
    # Calculate percentage removed
    percentage_removed = (total_phenotypes_removed / total_original_phenotypes * 100) if total_original_phenotypes > 0 else 0
    
    # Print summary
    if verbose:
        print("\n" + "=" * 50)
        print("PHENOTYPE REMOVAL SUMMARY")
        print("=" * 50)
        print(f"Total patients processed: {total_patients}")
        print(f"Patients with phenotypes removed: {patients_modified}")
        print(f"Total original phenotypes: {total_original_phenotypes}")
        print(f"Total phenotypes removed: {total_phenotypes_removed}")
        print(f"Percentage of phenotypes removed: {percentage_removed:.2f}%")
        print(f"Remaining phenotypes: {total_original_phenotypes - total_phenotypes_removed}")
        print("=" * 50)
    
    return modified_data


def save_cleaned_data(
    original_data: Dict[str, Any], 
    output_file: str, 
    verbose: bool = False
) -> None:
    """
    Process and save cleaned data to a JSON file
    
    Args:
        original_data: Original dataset
        output_file: Path to save the cleaned data
        verbose: Whether to print processing details
    """
    # Clean the data
    cleaned_data = remove_disease_phenotypes_from_dataset(original_data, verbose=verbose)
    
    # Save to file
    try:
        with open(output_file, 'w') as f:
            json.dump(cleaned_data, f, indent=2)
        print(f"\nCleaned data saved to: {output_file}")
    except Exception as e:
        print(f"Error saving cleaned data: {str(e)}")


# Example usage
if __name__ == "__main__":
    # Example of how to use the function
    from rdma.utils.data import read_json_file
    
    # Load data
    data = read_json_file("data/medical_students_data/high_agreement_phenotypes.json")
    
    # Remove disease phenotypes with verbose output
    cleaned_data = remove_disease_phenotypes_from_dataset(
        data=data,
        verbose=True
    )
    
    # Save cleaned data
    save_cleaned_data(
        original_data=data,
        output_file="data/medical_students_data/cleaned_high_agreement_with_phenotypes.json",
        verbose=True
    )

Processing 145 patients...
Total original phenotypes across all patients: 19008
Patient ID: 10877494
  Original phenotypes: 71
  Disease entities: ['angioimmunoblastic t-cell lymphoma', 'autoimmune hemolytic anemia', 'lymphoma']
  Removed phenotype: 'recently diagnosed Autoimmune Hemolytic anemia'
  Removed phenotype: 'autoimmune hemolytic anemia'
  Removed phenotype: 'lymphoma'
  Removed phenotype: 'anemia'
  Removed phenotype: 't-cell lymphoma'
  Removed phenotype: 'autoimmune hemolytic anemia'
  Total phenotypes removed: 6
  Remaining phenotypes: 65
  Final phenotypes: 65
------------------------------------------------------------
Patient ID: 10402135
  Original phenotypes: 147
  Disease entities: ['postpoliomyelitis syndrome']
  Final phenotypes: 147
------------------------------------------------------------
Patient ID: 10844136
  Original phenotypes: 249
  Disease entities: ['renal cell carcinoma', 'discoid lupus erythematosus']
  Removed phenotype: 'renal cell carcinoma'
  Rem

In [3]:
from rdma.utils.llm_client import LocalLLMClient

# llm_client = LocalLLMClient(model_type="mistral_24b", device="cuda:0",temperature=0.0001)
llm_client = LocalLLMClient(model_type="qwen_32b", device="cuda:7",temperature=0.0001)


Initialized ModelLoader with cache directory: /shared/rsaas/jw3/rare_disease/model_cache
Loading LLM!
Device configuration: cuda:7
Using device map: {'': 'cuda:7'}
Loading 70B model with quantization: qwen_32b
Generated cache path: /shared/rsaas/jw3/rare_disease/model_cache/Qwen3-32B_4bit_nf4
Valid cache found at /shared/rsaas/jw3/rare_disease/model_cache/Qwen3-32B_4bit_nf4
Loading cached quantized model from /shared/rsaas/jw3/rare_disease/model_cache/Qwen3-32B_4bit_nf4
{'': 'cuda:7'}
Qwen/Qwen3-32B


/home/johnwu3/miniconda3/envs/hporag/lib/python3.10/site-packages/transformers/quantizers/auto.py:239: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Model loaded!
Creating pipeline...
Detected Qwen model, using custom QwenPipeline
DEBUG: Applying chat template to 2 messages
DEBUG: Messages: [{'role': 'system', 'content': 'You are an expert and experienced from the healthcare and biomedical domain with extensive medical knowledge and practical experience. Your job is to annotate different medical contexts and answer related questions. Please answer the below message.'}, {'role': 'user', 'content': 'Hello?'}]
DEBUG: Chat template applied successfully with enable_thinking=True
QwenPipeline test output: <think>
Okay, the user just said "Hello?" I need to respond appropriately. Since they're in the healthcare domain, maybe they have a medical question. I should greet them back and ask how I can assist. Keep it friendly and open-ended


In [5]:
llm_client.pipeline("lets see")

DEBUG: Applying chat template to 1 messages
DEBUG: Messages: [{'role': 'user', 'content': 'lets see'}]
DEBUG: Chat template applied successfully with enable_thinking=True


[{'generated_text': "Hello! It looks like you're just starting out. How can I assist you today? Are you looking to explore a specific topic, solve a problem, or just want to have a conversation? Let me know, and I'll do my best to help! 😊"}]

In [4]:
llm_client.query("testing the Qwen model", "testing")

error here?75
error here???79, <|im_start|>system
testing<|im_end|>
<|im_start|>user
testing the Qwen model<|im_end|>
<|im_start|>assistant

error here?85
DEBUG: Applying chat template to 1 messages
DEBUG: Messages: [{'role': 'user', 'content': '<|im_start|>system\ntesting<|im_end|>\n<|im_start|>user\ntesting the Qwen model<|im_end|>\n<|im_start|>assistant\n'}]
DEBUG: Chat template applied successfully with enable_thinking=True


TypeError: 'NoneType' object cannot be interpreted as an integer

# Benchmarking + More Robust LLM Evaluation in 1 Script.  Hit @ 1 (can be expanded to Hit @ 10) for later.

In [ ]:
import json
import re
from typing import List, Dict, Tuple, Any
from rdma.utils.llm_client import LocalLLMClient

llm_client = LocalLLMClient(model_type="mistral_24b", device="cuda:0",temperature=0.0001)


def parse_diseases_list(response: str) -> List[str]:
    """
    Robust function to parse LLM response into Python list
    """
    try:
        # First, try direct JSON parsing
        cleaned = response.strip()
        diseases_list = json.loads(cleaned)
        
        # Validate it's a list
        if isinstance(diseases_list, list):
            return diseases_list
        else:
            return []
            
    except json.JSONDecodeError:
        # Try to extract JSON array using regex
        json_pattern = r'\[(.*?)\]'
        match = re.search(json_pattern, response, re.DOTALL)
        
        if match:
            try:
                json_content = '[' + match.group(1) + ']'
                diseases_list = json.loads(json_content)
                return diseases_list
            except json.JSONDecodeError:
                pass
        
        # Last resort: manual parsing for common formats
        try:
            # Remove brackets and split by comma
            content = response.strip()
            content = re.sub(r'^\[|\]$', '', content)  # Remove outer brackets
            
            # Split by comma and clean each item
            items = [item.strip().strip('"\'') for item in content.split(',')]
            items = [item for item in items if item]  # Remove empty items
            
            return items
            
        except Exception:
            return []

def normalize_disease_name(disease: str) -> str:
    """
    Normalize disease name for comparison: lowercase and strip whitespace
    """
    return disease.lower().strip()

def benchmark_rare_disease_diagnosis(
    data: Dict[str, Any], 
    llm_client: Any, 
    num_samples: int = None,
    verbose: bool = False
) -> Dict[str, float]:
    """
    Benchmark rare disease diagnosis performance
    
    Args:
        data: Dictionary containing patient data with phenotypes and disease entities
        llm_client: LLM client with query method
        num_samples: Number of samples to evaluate (None for all)
        verbose: Whether to print detailed results for each case
    
    Returns:
        Dictionary with benchmark metrics
    """
    
    # System prompt for LLM
    diff_diag_sys_prompt = """Given the following phenotypes, identify the top 10 most likely rare diseases.

CRITICAL: Your response must be EXACTLY in this JSON format with no additional text:

["Disease Name 1", "Disease Name 2", "Disease Name 3", "Disease Name 4", "Disease Name 5", "Disease Name 6", "Disease Name 7", "Disease Name 8", "Disease Name 9", "Disease Name 10"]

Rules:
1. Return ONLY the JSON array - no explanations, no additional text
2. Use double quotes around disease names
3. Separate diseases with commas
4. First disease = most likely, last disease = least likely
5. If you find fewer than 10 diseases, that's okay
6. If you cannot find any rare diseases, return: []

Example of correct format:
["Marfan Syndrome", "Ehlers-Danlos Syndrome", "Osteogenesis Imperfecta"]

Your response:"""
    
    # Initialize counters
    total_diseases = 0
    hits = 0
    hits_at_1 = 0
    total_patients = 0
    patients_with_hits = 0
    parsing_failures = 0
    
    # Get samples to process
    items_to_process = list(data.items())
    if num_samples is not None:
        items_to_process = items_to_process[:num_samples]
    
    if verbose:
        print(f"Evaluating {len(items_to_process)} patients...")
        print("=" * 60)
    
    for patient_id, patient_data in items_to_process:
        if 'matched_phenotypes' not in patient_data:
            continue
            
        total_patients += 1
        patient_hits = 0
        
        # Build phenotypes string
        phenotypes = ""
        for phenotype in patient_data['matched_phenotypes']:
            phenotypes += phenotype['phenotype'] + ", "
        
        # Query LLM
        phenotypes_prompt = f"Phenotypes: {phenotypes}"
        llm_response = llm_client.query(
            system_message=diff_diag_sys_prompt, 
            user_input=phenotypes_prompt
        )
        
        # Parse response
        predicted_diseases = parse_diseases_list(llm_response)
        
        if not predicted_diseases:
            parsing_failures += 1
            if verbose:
                print(f"Patient {patient_id}: Failed to parse LLM response")
                print(f"Raw response: '{llm_response}'")
            continue
        
        # Normalize predicted diseases for comparison
        predicted_normalized = [normalize_disease_name(d) for d in predicted_diseases]
        
        # Get ground truth diseases
        observed_diseases = patient_data.get('disease_entities', [])
        
        if verbose:
            print(f"Patient ID: {patient_id}")
            print(f"Phenotypes: {phenotypes.strip(', ')}")
            print(f"Predicted diseases: {predicted_diseases}")
            print(f"Observed diseases: {observed_diseases}")
        
        # Calculate hits for this patient
        for observed_disease in observed_diseases:
            observed_normalized = normalize_disease_name(observed_disease)
            total_diseases += 1
            
            # Check if disease is in top-10 predictions
            hit_in_top10 = observed_normalized in predicted_normalized
            if hit_in_top10:
                hits += 1
                patient_hits += 1
                
                if verbose:
                    print(f"  ✓ Hit: '{observed_disease}' found in predictions")
            else:
                if verbose:
                    print(f"  ✗ Miss: '{observed_disease}' not found in predictions")
            
            # Check if disease is the top prediction (Hit@1)
            hit_at_1 = (observed_normalized == predicted_normalized[0] 
                       if predicted_normalized else False)
            if hit_at_1:
                hits_at_1 += 1
                if verbose:
                    print(f"  ✓ Hit@1: '{observed_disease}' is top prediction")
        
        if patient_hits > 0:
            patients_with_hits += 1
        
        if verbose:
            print(f"  Patient hits: {patient_hits}/{len(observed_diseases)}")
            print("-" * 60)
    
    # Calculate final metrics
    hit_rate = hits / total_diseases if total_diseases > 0 else 0
    hit_at_1_rate = hits_at_1 / total_diseases if total_diseases > 0 else 0
    patient_hit_rate = patients_with_hits / total_patients if total_patients > 0 else 0
    parsing_success_rate = 1 - (parsing_failures / total_patients) if total_patients > 0 else 0
    
    results = {
        'hit_rate': hit_rate,
        'hit_at_1_rate': hit_at_1_rate,
        'patient_hit_rate': patient_hit_rate,
        'parsing_success_rate': parsing_success_rate,
        'total_diseases': total_diseases,
        'total_patients': total_patients,
        'hits': hits,
        'hits_at_1': hits_at_1,
        'patients_with_hits': patients_with_hits,
        'parsing_failures': parsing_failures
    }
    
    return results

def print_benchmark_results(results: Dict[str, float]) -> None:
    """
    Print benchmark results in a formatted way
    """
    print("\n" + "=" * 50)
    print("RARE DISEASE DIAGNOSIS BENCHMARK RESULTS")
    print("=" * 50)
    print(f"Total patients evaluated: {results['total_patients']}")
    print(f"Total diseases to predict: {results['total_diseases']}")
    print(f"LLM parsing success rate: {results['parsing_success_rate']:.2%}")
    print("-" * 50)
    print(f"Hit Rate (Top-10): {results['hit_rate']:.2%} ({results['hits']}/{results['total_diseases']})")
    print(f"Hit@1 Rate: {results['hit_at_1_rate']:.2%} ({results['hits_at_1']}/{results['total_diseases']})")
    print(f"Patient Hit Rate: {results['patient_hit_rate']:.2%} ({results['patients_with_hits']}/{results['total_patients']})")
    print("=" * 50)

# Example usage:
if __name__ == "__main__":
    # Example of how to use the function
    from rdma.utils.data import read_json_file
    
    # Load data
    data = read_json_file("data/medical_students_data/high_agreement_with_phenotypes.json")
    
    # Run benchmark on first 5 patients with verbose output
    results = benchmark_rare_disease_diagnosis(
        data=data, 
        llm_client=llm_client,  # Your LLM client
        num_samples=2,
        verbose=True
    )
    
    # Print results
    print_benchmark_results(results)

Evaluating 2 patients...
Patient ID: 10402135
Phenotypes: shortness of breath, history of COPD, dementia, HTN, Afib, gerd, polio, left ankle paralysis, dyspnea, increasing dyspnea, cough, wheezing, mild epigastric pain, confused from her baseline, hallucinations, swollen right leg, tender to palpation, elevated systolic blood pressure, elevated reticulocyte count, hypoxia, leukocytosis, hyponatremia, decreased lactate, b/l effusions, possible consolidation, elevated A/G Ratio, sweats, nausea, vomiting, diarrhea, constipation, dark tarry, frequency, hematuria, Former alcohol abuse, Delusional disorder, auditory hallucination, raynaud's phenomenon, Diverticulosis, Colonic adenoma, left bundle branch block, hypertension, anxiety, cognitive decline, urinary incontinence, urge, Hearing difficulty, depression, breast cancer, frail appearing, hard of hearing, scattered rhonchi, rales, rales, decreased breath sounds at bases, systolic murmur, 1+ edema ___ to knee, 2+ edema, tongue protrudes sy

In [ ]:
import json
import re
from typing import List, Dict, Tuple, Any, Optional
from rdma.utils.llm_client import LocalLLMClient


def parse_diseases_response(response: str) -> Tuple[str, List[str]]:
    """
    Parse LLM response into explanation and diseases list
    """
    try:
        # First, try direct JSON parsing
        cleaned = response.strip()
        response_data = json.loads(cleaned)
        
        # Extract explanation and diseases
        if isinstance(response_data, dict):
            explanation = response_data.get('explanation', '')
            diseases = response_data.get('diseases', [])
            return explanation, diseases
        else:
            return "", []
            
    except json.JSONDecodeError:
        # Try to extract JSON object using regex
        json_pattern = r'\{.*\}'
        match = re.search(json_pattern, response, re.DOTALL)
        
        if match:
            try:
                json_content = match.group(0)
                response_data = json.loads(json_content)
                explanation = response_data.get('explanation', '')
                diseases = response_data.get('diseases', [])
                return explanation, diseases
            except json.JSONDecodeError:
                pass
        
        # Fallback: try to find explanation and diseases separately
        try:
            # Look for explanation section
            explanation_match = re.search(r'"explanation":\s*"([^"]*)"', response, re.DOTALL)
            explanation = explanation_match.group(1) if explanation_match else ""
            
            # Look for diseases array
            diseases_match = re.search(r'"diseases":\s*\[(.*?)\]', response, re.DOTALL)
            if diseases_match:
                diseases_content = diseases_match.group(1)
                # Parse individual disease names
                diseases = [d.strip().strip('"\'') for d in diseases_content.split(',') if d.strip()]
                return explanation, diseases
            
            return explanation, []
            
        except Exception:
            return "", []


def normalize_disease_name(disease: str) -> str:
    """
    Normalize disease name for comparison: lowercase and strip whitespace
    """
    return disease.lower().strip()


def diagnose_single_patient(
    patient_id: str,
    patient_data: Dict[str, Any], 
    llm_client: Any,
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Run diagnosis on a single patient and return detailed results
    
    Args:
        patient_id: ID of the patient
        patient_data: Patient data containing phenotypes and disease entities
        llm_client: LLM client with query method
        verbose: Whether to print detailed results
    
    Returns:
        Dictionary with diagnosis results
    """
    
    # Updated system prompt for JSON output with explanation
    diff_diag_sys_prompt = """Given the following phenotypes, identify the top 10 most likely rare diseases.

CRITICAL: Your response must be EXACTLY in this JSON format with no additional text:

{
  "explanation": "Brief explanation of your reasoning process and how you arrived at these diagnoses based on the phenotypes",
  "diseases": ["Disease Name 1", "Disease Name 2", "Disease Name 3", "Disease Name 4", "Disease Name 5", "Disease Name 6", "Disease Name 7", "Disease Name 8", "Disease Name 9", "Disease Name 10"]
}

Rules:
1. Return ONLY the JSON object - no additional text before or after
2. Use double quotes around all strings
3. The "explanation" should be a brief reasoning about the phenotype pattern
4. The "diseases" array should list diseases in order of likelihood (most likely first)
5. If you find fewer than 10 diseases, that's okay
6. If you cannot find any rare diseases, return: {"explanation": "Unable to identify rare diseases from given phenotypes", "diseases": []}

Example of correct format:
{
  "explanation": "The combination of tall stature, arachnodactyly, and lens dislocation strongly suggests connective tissue disorders, particularly those affecting fibrillin-1.",
  "diseases": ["Marfan Syndrome", "Ehlers-Danlos Syndrome", "Homocystinuria"]
}

Your response:"""
    
    if 'matched_phenotypes' not in patient_data:
        return {
            'patient_id': patient_id,
            'error': 'No matched_phenotypes found in patient data',
            'explanation': '',
            'predicted_diseases': [],
            'observed_diseases': [],
            'hits': 0,
            'parsing_success': False
        }
    
    # Build phenotypes string
    phenotypes = ""
    for phenotype in patient_data['matched_phenotypes']:
        phenotypes += phenotype['phenotype'] + ", "
    
    # Query LLM
    phenotypes_prompt = f"Phenotypes: {phenotypes}"
    llm_response = llm_client.query(
        system_message=diff_diag_sys_prompt, 
        user_input=phenotypes_prompt
    )
    
    # Parse response
    explanation, predicted_diseases = parse_diseases_response(llm_response)
    
    parsing_success = bool(predicted_diseases) or bool(explanation)
    
    # Get ground truth diseases
    observed_diseases = patient_data.get('disease_entities', [])
    
    # Calculate hits for this patient
    hits = 0
    if predicted_diseases:
        predicted_normalized = [normalize_disease_name(d) for d in predicted_diseases]
        
        for observed_disease in observed_diseases:
            observed_normalized = normalize_disease_name(observed_disease)
            if observed_normalized in predicted_normalized:
                hits += 1
    
    result = {
        'patient_id': patient_id,
        'phenotypes': phenotypes.strip(', '),
        'explanation': explanation,
        'predicted_diseases': predicted_diseases,
        'observed_diseases': observed_diseases,
        'hits': hits,
        'total_diseases': len(observed_diseases),
        'parsing_success': parsing_success,
        'raw_llm_response': llm_response
    }
    
    if verbose:
        print(f"Patient ID: {patient_id}")
        print(f"Phenotypes: {result['phenotypes']}")
        print(f"Explanation: {explanation}")
        print(f"Predicted diseases: {predicted_diseases}")
        print(f"Observed diseases: {observed_diseases}")
        print(f"Hits: {hits}/{len(observed_diseases)}")
        if not parsing_success:
            print(f"⚠️  Parsing failed. Raw response: {llm_response}")
        print("-" * 60)
    
    return result


def benchmark_rare_disease_diagnosis(
    data: Dict[str, Any], 
    llm_client: Any, 
    num_samples: int = None,
    patient_id: str = None,
    verbose: bool = False
) -> Dict[str, Any]:
    """
    Benchmark rare disease diagnosis performance
    
    Args:
        data: Dictionary containing patient data with phenotypes and disease entities
        llm_client: LLM client with query method
        num_samples: Number of samples to evaluate (None for all)
        patient_id: Specific patient ID to evaluate (overrides num_samples)
        verbose: Whether to print detailed results for each case
    
    Returns:
        Dictionary with benchmark metrics and individual results
    """
    
    # Initialize counters
    total_diseases = 0
    hits = 0
    hits_at_1 = 0
    total_patients = 0
    patients_with_hits = 0
    parsing_failures = 0
    individual_results = []
    
    # Handle specific patient ID
    if patient_id:
        if patient_id not in data:
            return {
                'error': f'Patient ID {patient_id} not found in data',
                'available_patients': list(data.keys())[:10]  # Show first 10 available IDs
            }
        
        items_to_process = [(patient_id, data[patient_id])]
        if verbose:
            print(f"Evaluating specific patient: {patient_id}")
    else:
        # Get samples to process
        items_to_process = list(data.items())
        if num_samples is not None:
            items_to_process = items_to_process[:num_samples]
        
        if verbose:
            print(f"Evaluating {len(items_to_process)} patients...")
    
    if verbose:
        print("=" * 60)
    
    for pid, patient_data in items_to_process:
        if 'matched_phenotypes' not in patient_data:
            continue
            
        total_patients += 1
        
        # Get diagnosis for this patient
        diagnosis_result = diagnose_single_patient(
            patient_id=pid,
            patient_data=patient_data,
            llm_client=llm_client,
            verbose=verbose
        )
        
        individual_results.append(diagnosis_result)
        
        # Update counters
        if not diagnosis_result['parsing_success']:
            parsing_failures += 1
            continue
        
        predicted_diseases = diagnosis_result['predicted_diseases']
        observed_diseases = diagnosis_result['observed_diseases']
        patient_hits = diagnosis_result['hits']
        
        # Update overall metrics
        total_diseases += len(observed_diseases)
        hits += patient_hits
        
        if patient_hits > 0:
            patients_with_hits += 1
        
        # Check Hit@1
        if predicted_diseases and observed_diseases:
            predicted_normalized = [normalize_disease_name(d) for d in predicted_diseases]
            for observed_disease in observed_diseases:
                observed_normalized = normalize_disease_name(observed_disease)
                if predicted_normalized and observed_normalized == predicted_normalized[0]:
                    hits_at_1 += 1
    
    # Calculate final metrics
    hit_rate = hits / total_diseases if total_diseases > 0 else 0
    hit_at_1_rate = hits_at_1 / total_diseases if total_diseases > 0 else 0
    patient_hit_rate = patients_with_hits / total_patients if total_patients > 0 else 0
    parsing_success_rate = 1 - (parsing_failures / total_patients) if total_patients > 0 else 0
    
    results = {
        'hit_rate': hit_rate,
        'hit_at_1_rate': hit_at_1_rate,
        'patient_hit_rate': patient_hit_rate,
        'parsing_success_rate': parsing_success_rate,
        'total_diseases': total_diseases,
        'total_patients': total_patients,
        'hits': hits,
        'hits_at_1': hits_at_1,
        'patients_with_hits': patients_with_hits,
        'parsing_failures': parsing_failures,
        'individual_results': individual_results
    }
    
    return results


def print_benchmark_results(results: Dict[str, Any]) -> None:
    """
    Print benchmark results in a formatted way
    """
    if 'error' in results:
        print(f"Error: {results['error']}")
        if 'available_patients' in results:
            print(f"Available patient IDs (first 10): {results['available_patients']}")
        return
    
    print("\n" + "=" * 50)
    print("RARE DISEASE DIAGNOSIS BENCHMARK RESULTS")
    print("=" * 50)
    print(f"Total patients evaluated: {results['total_patients']}")
    print(f"Total diseases to predict: {results['total_diseases']}")
    print(f"LLM parsing success rate: {results['parsing_success_rate']:.2%}")
    print("-" * 50)
    print(f"Hit Rate (Top-10): {results['hit_rate']:.2%} ({results['hits']}/{results['total_diseases']})")
    print(f"Hit@1 Rate: {results['hit_at_1_rate']:.2%} ({results['hits_at_1']}/{results['total_diseases']})")
    print(f"Patient Hit Rate: {results['patient_hit_rate']:.2%} ({results['patients_with_hits']}/{results['total_patients']})")
    print("=" * 50)


def print_individual_results(results: Dict[str, Any], show_explanations: bool = True) -> None:
    """
    Print individual patient results
    """
    if 'individual_results' not in results:
        print("No individual results available")
        return
    
    print("\n" + "=" * 50)
    print("INDIVIDUAL PATIENT RESULTS")
    print("=" * 50)
    
    for result in results['individual_results']:
        print(f"\nPatient ID: {result['patient_id']}")
        print(f"Phenotypes: {result['phenotypes']}")
        
        if show_explanations and result['explanation']:
            print(f"LLM Explanation: {result['explanation']}")
        
        print(f"Predicted diseases: {result['predicted_diseases']}")
        print(f"Observed diseases: {result['observed_diseases']}")
        print(f"Hits: {result['hits']}/{result['total_diseases']}")
        
        if not result['parsing_success']:
            print("⚠️  Parsing failed")
        
        print("-" * 40)


# Example usage:
if __name__ == "__main__":
    # Example of how to use the function
    from rdma.utils.data import read_json_file
    
    # Load data
    data = read_json_file("data/medical_students_data/high_agreement_with_phenotypes.json")
    
    # Example 1: Run benchmark on specific patient
    specific_patient_id = "your_patient_id_here"  # Replace with actual patient ID
    results_specific = benchmark_rare_disease_diagnosis(
        data=data, 
        llm_client=llm_client,
        patient_id=specific_patient_id,
        verbose=True
    )
    
    print_benchmark_results(results_specific)
    print_individual_results(results_specific, show_explanations=True)
    
    print("\n" + "="*80 + "\n")
    
    # Example 2: Run benchmark on first 2 patients with verbose output
    results_sample = benchmark_rare_disease_diagnosis(
        data=data, 
        llm_client=llm_client,
        num_samples=2,
        verbose=True
    )
    
    print_benchmark_results(results_sample)
    print_individual_results(results_sample, show_explanations=True)
    
    # Example 3: Diagnose single patient directly
    patient_ids = list(data.keys())
    if patient_ids:
        single_result = diagnose_single_patient(
            patient_id=patient_ids[0],
            patient_data=data[patient_ids[0]],
            llm_client=llm_client,
            verbose=True
        )

Error: Patient ID your_patient_id_here not found in data
Available patient IDs (first 10): ['10877494', '10402135', '10844136', '10948957', '10981539', '10982872', '11208462', '11369570', '11561996', '11960419']
No individual results available


Evaluating 2 patients...
Patient ID: 10877494
Phenotypes: recently diagnosed Autoimmune Hemolytic anemia, lymphadenopathy, with possible underlying malignancy, generalized, weakness, dyspena, + 10 lbs weight, pain, palpitations, cough, abd pain, n/v/d/c, dysuria, focal weakness, rash, baseline numbness in left hand, hypertension, hyperlipidemia, autoimmune hemolytic anemia, prostate cancer, O2 95%RA, splenomegaly, lymphadenopathies, anterior medistinal lymph node of 3 cm, gallbladder with stones, duodenal diverticula, Diverticulosis, Small hiatal hernia, splenic cyst, elevated glucose, decreased total CO2, abnormal anion gap, elevated total bilirubin, elevated uric acid, elevated ferritin, increased hct, increased mchc, prolonged ptt, pneumonia

In [6]:
import random

def sample_patients(data, seed=None):
    """
    Sample 3 patients with hits@1 and 3 patients with no hits.
    
    Args:
        data (dict): A dictionary containing patient IDs and their hit results.
        seed (int, optional): Random seed for reproducible sampling.
        
    Returns:
        list: A list of patient IDs [hits@1_patients, no_hits_patients]
    """
    if seed is not None:
        random.seed(seed)
    
    # Separate patients into categories
    hits_at_1_patients = []
    no_hits_patients = []
    
    for patient_id, hits in data.items():
        if hits.get("1", False):
            hits_at_1_patients.append(patient_id)
        elif not hits.get("10", False):  # No hits at any level (using same logic as your original code)
            no_hits_patients.append(patient_id)
    
    # Sample 3 from each category (or all available if less than 3)
    sampled_hits_at_1 = random.sample(hits_at_1_patients, min(3, len(hits_at_1_patients)))
    sampled_no_hits = random.sample(no_hits_patients, min(3, len(no_hits_patients)))
    
    # Return combined list
    return sampled_hits_at_1 + sampled_no_hits

# Example usage
if __name__ == "__main__":
    # Load the JSON data from the string
    patient_data = read_json_file("data/medical_students_data/mistral_hits.json")
    
    # Sample patients
    sampled_patient_ids = sample_patients(patient_data, seed=42)  # Using seed for reproducibility
    
    print(f"Sampled patient IDs: {sampled_patient_ids}")
    print(f"Total sampled: {len(sampled_patient_ids)}")

Sampled patient IDs: ['13740941', '11369570', '18235940', '11854321', '13106750', '17454400']
Total sampled: 6


In [10]:
import json
import pandas as pd
from typing import List, Dict, Any, Optional
from rdma.utils.data import read_json_file
from rdma.utils.llm_client import LocalLLMClient

# Import the functions from your diagnosis module


def rebenchmark_patients_with_export(
    patient_ids: List[str],
    data_file_path: str,
    llm_client: Any,
    output_csv_path: str = "patient_diagnosis_results.csv",
    verbose: bool = True
) -> Dict[str, Any]:
    """
    Rebenchmark specific patients and export results to CSV and return as dictionary.
    
    Args:
        patient_ids: List of patient IDs to process
        data_file_path: Path to the JSON file containing patient data
        llm_client: LLM client for making predictions
        output_csv_path: Path to save the CSV file
        verbose: Whether to print progress information
    
    Returns:
        Dictionary containing all results and export data
    """
    
    # Load patient data
    if verbose:
        print(f"Loading patient data from {data_file_path}...")
    
    try:
        data = read_json_file(data_file_path)
    except Exception as e:
        print(f"Error loading data file: {e}")
        return {"error": f"Failed to load data file: {e}"}
    
    # Initialize results storage
    export_data = []
    results_dict = {
        "metadata": {
            "total_requested_patients": len(patient_ids),
            "successfully_processed": 0,
            "failed_patients": [],
            "missing_patients": []
        },
        "individual_results": [],
        "export_data": []
    }
    
    if verbose:
        print(f"Processing {len(patient_ids)} patients...")
        print("=" * 60)
    
    for i, patient_id in enumerate(patient_ids, 1):
        if verbose:
            print(f"Processing patient {i}/{len(patient_ids)}: {patient_id}")
        
        # Check if patient exists in data
        if patient_id not in data:
            if verbose:
                print(f"⚠️  Patient {patient_id} not found in data")
            results_dict["metadata"]["missing_patients"].append(patient_id)
            continue
        
        try:
            # Get diagnosis for this patient
            diagnosis_result = diagnose_single_patient(
                patient_id=patient_id,
                patient_data=data[patient_id],
                llm_client=llm_client,
                verbose=False  # We'll handle our own verbose output
            )
            
            if not diagnosis_result.get('parsing_success', False):
                if verbose:
                    print(f"⚠️  Failed to parse LLM response for patient {patient_id}")
                results_dict["metadata"]["failed_patients"].append(patient_id)
                continue
            
            # Format phenotypes as a clean string
            phenotypes_str = diagnosis_result.get('phenotypes', '')
            
            # Format predictions as a clean string
            predictions = diagnosis_result.get('predicted_diseases', [])
            predictions_str = "; ".join(predictions) if predictions else ""
            
            # Get explanation
            explanation = diagnosis_result.get('explanation', '')
            
            # Format ground truth diseases
            ground_truth = diagnosis_result.get('observed_diseases', [])
            ground_truth_str = "; ".join(ground_truth) if ground_truth else ""
            
            # Calculate hit diseases (correct predictions)
            hit_diseases = []
            if predictions and ground_truth:
                predicted_normalized = [normalize_disease_name(d) for d in predictions]
                for observed_disease in ground_truth:
                    observed_normalized = normalize_disease_name(observed_disease)
                    if observed_normalized in predicted_normalized:
                        hit_diseases.append(observed_disease)
            
            hit_diseases_str = "; ".join(hit_diseases) if hit_diseases else ""
            
            # Create export row
            export_row = {
                "patient_id": patient_id,
                "phenotypes": phenotypes_str,
                "prediction": predictions_str,
                "explanation": explanation,
                "ground_truth_rare_diseases": ground_truth_str,
                "hit_diseases": hit_diseases_str,
                "num_hits": diagnosis_result.get('hits', 0),
                "total_ground_truth": diagnosis_result.get('total_diseases', 0)
            }
            
            export_data.append(export_row)
            results_dict["individual_results"].append(diagnosis_result)
            results_dict["metadata"]["successfully_processed"] += 1
            
            if verbose:
                print(f"✓ Successfully processed patient {patient_id}")
                print(f"  Phenotypes: {phenotypes_str[:100]}{'...' if len(phenotypes_str) > 100 else ''}")
                print(f"  Predictions: {predictions_str[:100]}{'...' if len(predictions_str) > 100 else ''}")
                print(f"  Ground truth: {ground_truth_str}")
                print(f"  Hit diseases: {hit_diseases_str}")
                print(f"  Hits: {diagnosis_result.get('hits', 0)}/{diagnosis_result.get('total_diseases', 0)}")
                print()
            
        except Exception as e:
            if verbose:
                print(f"❌ Error processing patient {patient_id}: {str(e)}")
            results_dict["metadata"]["failed_patients"].append(patient_id)
            continue
    
    # Save to CSV
    if export_data:
        try:
            df = pd.DataFrame(export_data)
            df.to_csv(output_csv_path, index=False)
            if verbose:
                print(f"✓ Results exported to {output_csv_path}")
                print(f"  Exported {len(export_data)} patient records")
        except Exception as e:
            print(f"❌ Error saving CSV file: {e}")
            results_dict["csv_export_error"] = str(e)
    else:
        print("⚠️  No data to export - no patients were successfully processed")
    
    # Add export data to results dictionary
    results_dict["export_data"] = export_data
    
    # Print summary
    if verbose:
        print("\n" + "=" * 60)
        print("PROCESSING SUMMARY")
        print("=" * 60)
        print(f"Total requested patients: {results_dict['metadata']['total_requested_patients']}")
        print(f"Successfully processed: {results_dict['metadata']['successfully_processed']}")
        print(f"Missing patients: {len(results_dict['metadata']['missing_patients'])}")
        print(f"Failed patients: {len(results_dict['metadata']['failed_patients'])}")
        
        if results_dict["metadata"]["missing_patients"]:
            print(f"Missing patient IDs: {results_dict['metadata']['missing_patients']}")
        
        if results_dict["metadata"]["failed_patients"]:
            print(f"Failed patient IDs: {results_dict['metadata']['failed_patients']}")
        
        print("=" * 60)
    
    return results_dict



# Example usage functions
def main_example():
    """Example of how to use the export functions"""
    
    # Initialize your LLM client    
    # Example 1: Export specific patient IDs
    specific_patient_ids = ['13740941', '11369570', '18235940', '11854321', '13106750', '17454400']  # Replace with actual IDs
    
    results = rebenchmark_patients_with_export(
        patient_ids=specific_patient_ids,
        data_file_path="data/medical_students_data/high_agreement_with_phenotypes.json",
        llm_client=llm_client,
        output_csv_path="specific_patients_diagnosis.csv",
        verbose=True
    )
    

    
    return results

if __name__ == "__main__":
    # Run the example
    results  = main_example()
    
    # Access the results
    print(f"Specific patients results: {len(results.get('export_data', []))} records")

Loading patient data from data/medical_students_data/high_agreement_with_phenotypes.json...


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processing 6 patients...
Processing patient 1/6: 13740941
✓ Successfully processed patient 13740941
  Phenotypes: Abrupt-onset headaches, mitochondrial disorder, headache, GI illness, nausea, postprandial abdominal...
  Predictions: Reversible Cerebral Vasoconstriction Syndrome; Mitochondrial Encephalomyopathy, Lactic Acidosis, and...
  Ground truth: reversible cerebral vasoconstriction syndrome; rcvs
  Hit diseases: reversible cerebral vasoconstriction syndrome
  Hits: 1/2

Processing patient 2/6: 11369570
✓ Successfully processed patient 11369570
  Phenotypes: Allergies, dyspnea on exertion, tia, infarcts in right cerebral cortex, fibrillation, history of hyp...
  Predictions: Multiple Myeloma; Amyloidosis; Waldenstrom Macroglobulinemia; POEMS Syndrome; Light Chain Deposition...
  Ground truth: multiple myeloma
  Hit diseases: multiple myeloma
  Hits: 1/1

Processing patient 3/6: 18235940
✓ Successfully processed patient 18235940
  Phenotypes: renal mass, R renal cyst, back pain, enl